# 02.2 CNN 基础（CNN Basics）

这一节开始真正进入卷积神经网络  

重点概念

- 卷积层（convolution layer）
- 卷积核（kernel）
- 步幅（stride）
- 填充（padding）
- 池化（pooling）
- 特征图（feature map）
- 通道数变化（channel changes）

## 学习目标

学完后你应该能

1. 理解 `Conv2d` 的输入输出结构
2. 解释卷积后 shape 如何变化
3. 理解 `stride` 和 `padding` 的作用
4. 理解池化层
5. 读懂一个小型 CNN 的 shape 流动
6. 为后面的图像分类实战做好结构准备

In [ ]:
import torch
import torch.nn as nn

## 1. `Conv2d` 的输入和输出

`Conv2d` 的输入通常是：  

- `(N, C, H, W)`

其中

- `C`：输入通道数（number of input channels）
- `H`：高度（height）
- `W`：宽度（width）

输出通常是：  

- `(N, C_out, H_out, W_out)`

In [ ]:
x = torch.randn(4, 1, 8, 8)
conv = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, stride=1, padding=1)
y = conv(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)

这里的输出 shape 是 `(4, 3, 8, 8)`，因为：  

- batch size 不变
- 输出通道数由 `out_channels=3` 决定
- `padding=1` 且 `stride=1` 时，空间尺寸保持不变（with `padding=1` and `stride=1`, spatial size stays unchanged）

## 2. `kernel_size`、`stride`、`padding`
## `kernel_size`, `stride`, and `padding`

feature map 的空间大小。  

先建立直觉

- `kernel_size` 大一点：看得更大块
- `stride` 大一点：走得更快，输出更小（larger jumps, smaller output）
- `padding` 大一点：保留边界信息更多（preserves more border information）

In [ ]:
x = torch.randn(1, 1, 8, 8)

conv_a = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=0)
conv_b = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
conv_c = nn.Conv2d(1, 4, kernel_size=3, stride=2, padding=1)

print("input shape =", x.shape)
print("padding=0, stride=1 ->", conv_a(x).shape)
print("padding=1, stride=1 ->", conv_b(x).shape)
print("padding=1, stride=2 ->", conv_c(x).shape)

## 3. 参数量

卷积层也是有可学习参数的。  

对 `Conv2d(in_channels=C_in, out_channels=C_out, kernel_size=k)`：  

- 权重参数量
- 如果有 bias，还要再加 `C_out`（add another `C_out` if bias is used）

In [ ]:
conv = nn.Conv2d(1, 3, kernel_size=3)

num_params = sum(p.numel() for p in conv.parameters())
print("num_params =", num_params)

# 手算
# weights = 3 * 1 * 3 * 3 = 27
# bias = 3
# total = 30

In [ ]:
# 练习 1
# 假设有一层
# Conv2d(in_channels=3, out_channels=8, kernel_size=5)
#
# 请手算总参数量，并用代码验证。
# Manually compute the total number of parameters, then verify with code.

# conv_ex =
# print(sum(p.numel() for p in conv_ex.parameters()))

In [ ]:
# 练习 1 参考答案

conv_ex = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=5)
print(sum(p.numel() for p in conv_ex.parameters()))

# weights = 8 * 3 * 5 * 5 = 600
# bias = 8
# total = 608

## 4. 池化层

pooling 常用于减少空间尺寸。  

最常见的是

- `MaxPool2d`

它的直觉是：在一个小窗口里取最大值。  


In [ ]:
x = torch.randn(2, 4, 8, 8)
pool = nn.MaxPool2d(kernel_size=2, stride=2)
y = pool(x)

print("x.shape =", x.shape)
print("y.shape =", y.shape)

这里从 `8x8` 变成 `4x4`，因为 `2x2` 池化窗口配合 `stride=2` 会把高宽都减半。  


## 5. 一个最小 CNN 的 shape 流动

读 CNN 代码时，最重要的是一直跟踪 shape。  


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 2 * 2, 10)

    def forward(self, x):
        print("input:", x.shape)
        x = self.conv1(x)
        print("after conv1:", x.shape)
        x = self.relu1(x)
        x = self.pool1(x)
        print("after pool1:", x.shape)
        x = self.conv2(x)
        print("after conv2:", x.shape)
        x = self.relu2(x)
        x = self.pool2(x)
        print("after pool2:", x.shape)
        x = x.flatten(start_dim=1)
        print("after flatten:", x.shape)
        x = self.fc(x)
        print("after fc:", x.shape)
        return x


model = TinyCNN()
dummy = torch.randn(4, 1, 8, 8)
out = model(dummy)

你要特别注意这条 shape 链：  

- `(4, 1, 8, 8)`
- `(4, 8, 8, 8)`
- `(4, 8, 4, 4)`
- `(4, 16, 4, 4)`
- `(4, 16, 2, 2)`
- `(4, 64)`
- `(4, 10)`

这就是 CNN 代码阅读的基本功。  


In [ ]:
# 练习 2
# 如果输入是 (batch, 1, 8, 8)，经过：
# If the input is (batch, 1, 8, 8), and it passes through:
# 1. Conv2d(1, 4, kernel_size=3, padding=1)
# 2. MaxPool2d(2)
#
# 输出 shape 是多少？
# What is the output shape?
#
# 请先手算，再用代码验证。
# Compute it by hand first, then verify with code.

# x =
# conv =
# pool =
# y =
# print(y.shape)

In [ ]:
# 练习 2 参考答案

x = torch.randn(5, 1, 8, 8)
conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
pool = nn.MaxPool2d(2)
y = pool(conv(x))
print(y.shape)

# shape: (5, 4, 4, 4)

## 6. `Flatten` 和全连接层

卷积层输出通常还是四维张量。  

linear layer，通常需要先 `flatten`。  


In [ ]:
x = torch.randn(3, 16, 2, 2)
x_flat = x.flatten(start_dim=1)

print("x.shape =", x.shape)
print("x_flat.shape =", x_flat.shape)

`16 * 2 * 2 = 64`，所以 flatten 后变成 `(batch_size, 64)`。  


In [ ]:
# 练习 3
# 如果某层输出 shape 是 (7, 12, 3, 3)，
# If a layer output has shape (7, 12, 3, 3),
# flatten(start_dim=1) 之后 shape 是多少？
# what is the shape after flatten(start_dim=1)?

参考回答

`12 * 3 * 3 = 108`，所以 shape 是 `(7, 108)`。  


## 7. 小结

这节最重要的是建立 CNN 的 shape 直觉。  

你现在应该能回答

1. `Conv2d` 的输入和输出 shape 一般长什么样？
2. `out_channels` 为什么会改变特征图通道数？
3. 为什么池化层常常会让高宽变小？
4. 为什么卷积输出接全连接层前常要 `flatten`？

下一步建议

- 进入 CNN 图像分类实战 notebook，把这些结构真正跑起来